# Vectorless RAG with PageIndex

This notebook demonstrates how to implement a RAG system using [PageIndex](https://pageindex.ai/), a vectorless retrieval framework. Unlike traditional RAG, which relies on vector embeddings and similarity searches, PageIndex builds a hierarchical tree structure of your documents. This allows the system to use reasoning to navigate the document's natural organization (chapters, sections, and pages) to find the most relevant information.

In [4]:
!pip install pageindex

import os
import time
from getpass import getpass
from pageindex import PageIndexClient

# Initialize the PageIndex client
# You can get an API key from https://dash.pageindex.ai/
api_key = os.getenv("PAGEINDEX_API_KEY")
if not api_key:
    api_key = getpass("Enter your PageIndex API key: ")
    os.environ["PAGEINDEX_API_KEY"] = api_key

client = PageIndexClient(api_key=api_key)

## 1. Upload and Index the Document

We will use a sample PDF file to demonstrate the indexing process. PageIndex will parse the PDF and build a hierarchical tree structure.

In [5]:
import urllib.request

# Path to the local PDF file
pdf_path = "sample.pdf"

# Download the sample PDF if it is not present (e.g., in Google Colab)
if not os.path.exists(pdf_path):
    url = "https://raw.githubusercontent.com/bonigarcia/context-engineering/main/ch03/python/vectorless-rag-pageindex/sample.pdf"
    print(f"Downloading {pdf_path} from {url}...")
    try:
        urllib.request.urlretrieve(url, pdf_path)
    except Exception as e:
        print(f"Error downloading the file: {e}")

if not os.path.exists(pdf_path):
    print(f"Error: {pdf_path} not found. Please ensure it is in the same directory as this notebook.")
else:
    print(f"--- Indexing document: {pdf_path} ---")

    # 1. Upload and index the document
    # submit_document returns a dictionary containing the doc_id
    upload_result = client.submit_document(pdf_path)
    doc_id = upload_result.get("doc_id")

    print(f"Document submitted. ID: {doc_id}")
    print("Waiting for indexing to complete...")

    # Poll for completion status
    while True:
        doc_info = client.get_document(doc_id)
        status = doc_info.get("status")
        if status == "completed":
            break
        elif status == "failed":
            print("Indexing failed.")
            break
        time.sleep(2)

    if status == "completed":
        print("Indexing complete.")

--- Indexing document: sample.pdf ---
Document submitted. ID: pi-cmosrkt3e09s401qr96304xzo
Waiting for indexing to complete...
Indexing complete.


## 2. Reasoning-based Retrieval and Generation

Once the document is indexed, we can query it. PageIndex will use its hierarchical index to find the answer.

In [6]:
query = "What happens if the same note is changed on two different devices?"
print(f"Query: {query}")

# chat_completions performs reasoning-based retrieval and answer generation
# It follows an interface similar to OpenAI's chat completions
response = client.chat_completions(
    messages=[{"role": "user", "content": query}],
    doc_id=doc_id
)

print("\n--- Answer ---")
answer = response["choices"][0]["message"]["content"]
print(answer)

Query: What happens if the same note is changed on two different devices?

--- Answer ---
If the same note is edited on two different devices, **LumaNote will ask the user which version to keep** when connectivity is restored. This conflict resolution happens during the reconciliation process where the app compares local edits with the cloud copy after coming back online.


## 3. Optional: Retrieve Tree Structure

We can also retrieve the tree structure that PageIndex built for the document.

In [7]:
# Retrieve the tree structure or reasoning details if needed
tree = client.get_tree(doc_id)
print(f"Document Tree structure retrieved: {len(tree)} nodes found.")

Document Tree structure retrieved: 5 nodes found.
